In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 데이터 불러오기
df = pd.read_csv('../data/raw/ecommerce_customer_data_large.csv')

# 2. 날짜 데이터를 Datetime 형식으로 변환
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

# 3. 고객 단위(Customer ID)로 데이터 압축 (RFM 및 정적 정보 추출)
# 딕셔너리 형태로 각 컬럼을 어떻게 뭉칠지 정의합니다.
aggregations = {
    'Purchase Date': 'max',           # 가장 최근 구매일 (나중에 Recency 계산용)
    'Customer ID': 'count',           # 구매 횟수 (Frequency)
    'Total Purchase Amount': 'sum',   # 총 구매 금액 (Monetary)
    'Returns': 'sum',                 # 총 반품 횟수
    'Customer Age': 'first',          # 나이는 변하지 않으므로 첫 번째 값 사용
    'Gender': 'first',                # 성별도 첫 번째 값 사용
    'Churn': 'first'                  # 이탈 여부도 고객당 고정값이므로 첫 번째 값 사용
}

# groupby를 이용해 고객 단위로 병합
customer_df = df.groupby('Customer ID').agg(aggregations)

# 컬럼명 보기 좋게 변경
customer_df.rename(columns={
    'Purchase Date': 'Last_Purchase_Date',
    'Customer ID': 'Frequency',
    'Total Purchase Amount': 'Monetary'
}, inplace=True)

print("고객 단위 병합 완료\n데이터 크기:", customer_df.shape)
display(customer_df.head())

# ---------------------------------------------------------
# 4. 이제 안전하게 학습용 / 테스트용 데이터로 분리!
# ---------------------------------------------------------

X = customer_df.drop(columns=['Churn'])
y = customer_df['Churn']

# 여기서 분리해야 특정 고객의 정보가 한쪽으로만 깔끔하게 들어갑니다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n학습용 데이터 크기:", X_train.shape)
print("테스트용 데이터 크기:", X_test.shape)

고객 단위 병합 완료
데이터 크기: (49661, 7)


,Last_Purchase_Date,Frequency,Monetary,Returns,Customer Age,Gender,Churn
Customer ID,,,,,,,
1,2022-11-29 06:48:25,3,6290,0.0,67,Female,0
2,2023-07-03 17:26:19,6,16481,4.0,42,Female,0
3,2023-02-03 03:58:07,4,9423,0.0,31,Male,0
4,2022-06-29 03:41:09,5,7826,3.0,37,Male,0
5,2022-07-16 04:08:09,5,9769,3.0,24,Female,0



학습용 데이터 크기: (39728, 6)
테스트용 데이터 크기: (9933, 6)


In [3]:
# 1. Recency 계산 (기준일로부터 며칠 지났는지)
reference_date = customer_df['Last_Purchase_Date'].max() + pd.Timedelta(days=1)
customer_df['Recency'] = (reference_date - customer_df['Last_Purchase_Date']).dt.days

# 2. Gender 수치화 (Female: 0, Male: 1)
customer_df['Gender'] = customer_df['Gender'].map({'Female': 0, 'Male': 1})

# 3. 불필요해진 날짜 컬럼 삭제
customer_df.drop(columns=['Last_Purchase_Date'], inplace=True)

# 데이터 숫자화 완료
print("변환 완료")
display(customer_df.head())

변환 완료


,Frequency,Monetary,Returns,Customer Age,Gender,Churn,Recency
Customer ID,,,,,,,
1,3,6290,0.0,67,0,0,289
2,6,16481,4.0,42,0,0,73
3,4,9423,0.0,31,1,0,223
4,5,7826,3.0,37,1,0,442
5,5,9769,3.0,24,0,0,425


In [4]:
import os
import json

# 저장할 폴더 생성 (없을 경우)
os.makedirs('../data/processed', exist_ok=True)

# 1. 가공된 전체 데이터 저장
customer_df.to_csv('../data/processed/processed_customer_data.csv', index=True)

# 2. 메타데이터(통계치) 추출 및 JSON 저장
metadata = {
    "total_customers": int(customer_df.shape[0]),
    "churn_rate": float(customer_df['Churn'].mean()),
    "avg_monetary": float(customer_df['Monetary'].mean()),
    "avg_frequency": float(customer_df['Frequency'].mean()),
    "avg_recency": float(customer_df['Recency'].mean())
}

with open('../data/processed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("1일차 데이터 전처리 및 저장")

1일차 데이터 전처리 및 저장


In [5]:
print(customer_df[['Recency', 'Frequency', 'Monetary']].skew())

Recency      1.499599
Frequency    0.513827
Monetary     0.619205
dtype: float64


In [6]:
import numpy as np

# 1. Recency 로그 변환 (0이 있을 수 있으므로 log1p 사용)
customer_df['Recency_log'] = np.log1p(customer_df['Recency'])

# 2. 변환 후 왜도 다시 확인 (수치가 0에 가까워졌는지 보세요!)
print("변환 후 Recency 왜도:", customer_df['Recency_log'].skew())

# 3. 최종 데이터 저장 (이전에 짠 저장 코드 다시 실행)
customer_df.to_csv('../data/processed/processed_customer_data.csv', index=True)

변환 후 Recency 왜도: -0.8600798264782402


In [7]:
# 1. 최종 가공 데이터 저장 (로그 변환된 버전 포함)
customer_df.to_csv('../data/processed/processed_customer_data.csv', index=True)

# 2. 메타데이터 업데이트 (로그 변환 여부 기록)
metadata["has_log_transform"] = True
metadata["recency_skew_before"] = 1.49
metadata["recency_skew_after"] = -0.86

with open('../data/processed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("1일차 완료")

1일차 완료


In [8]:
import os
from dotenv import load_dotenv
import mlflow

# 환경 변수(.env) 로드
env_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'backend', '.env'))
load_dotenv(dotenv_path=env_path)

mlflow.set_tracking_uri("sqlite:///mlflow.db")

# 프로젝트 실험 공간 생성
experiment_name = "E-commerce_Churn_Prediction"
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(experiment_name)
mlflow.set_experiment(experiment_name)

print("1일차 수정 완료.")

1일차 수정 완료.
